# OpenRouter model usage rankings → `openrouter_rankings`

**Weekly** job. Re-pulls the full **daily** top-50-models-by-token-usage history (back to 2025-01-01)
and **overwrites** `openrouter_rankings`. OpenRouter retains and re-serves the whole history, so a
weekly overwrite is safe — nothing to append or lose. Roll up to week/month in SQL/Genie as needed.

**Prereq:** secret `OPENROUTER_KEY` (an `sk-or-v1…` key) in scope `frontier_labs`.
Limits: 30 req/min, 500/day; max 366 days per request (chunked below).

In [ ]:
CATALOG, SCHEMA, SCOPE, OR_SECRET = "fso_market_intelligence", "frontier_labs", "frontier_labs", "OPENROUTER_KEY"
TABLE = f"{CATALOG}.{SCHEMA}.openrouter_rankings"
BASE = "https://openrouter.ai/api/v1/datasets/rankings-daily"
HISTORY_START = "2025-01-01"

import requests, time
from datetime import date, timedelta
OR_KEY = dbutils.secrets.get(SCOPE, OR_SECRET)
H = {"Authorization": f"Bearer {OR_KEY}", "Accept": "application/json"}

def chunks(start, end, maxdays=365):
    cur = start
    while cur <= end:
        ce = min(cur + timedelta(days=maxdays), end)
        yield cur, ce
        cur = ce + timedelta(days=1)

rows = []
for s, e in chunks(date.fromisoformat(HISTORY_START), date.today()):
    r = requests.get(BASE, headers=H, params={"start_date": s.isoformat(), "end_date": e.isoformat(), "period": "day"}, timeout=90)
    r.raise_for_status()
    batch = r.json().get("data", [])
    rows += batch
    print(f"  {s} → {e}: +{len(batch)} ({len(rows)} total)")
    time.sleep(2)
print("rows:", len(rows))

In [ ]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType

sch = StructType([StructField("date", StringType()), StructField("model_permaslug", StringType()),
                  StructField("total_tokens", StringType())])
df = spark.createDataFrame(
    [{"date": r["date"], "model_permaslug": r["model_permaslug"], "total_tokens": str(r["total_tokens"])} for r in rows],
    schema=sch)

# aggregate 'other' row has no provider prefix — flag it and rank real models by daily tokens
df = (df.withColumn("date", F.to_date("date"))
        .withColumn("total_tokens", F.col("total_tokens").cast("long"))
        .withColumn("is_aggregate", ~F.col("model_permaslug").contains("/"))
        .withColumn("period", F.lit("day")))
w = Window.partitionBy("date").orderBy(F.col("total_tokens").desc())
df = df.withColumn("rank", F.row_number().over(w)).withColumn("captured_at", F.current_date())

(df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE))
t = spark.table(TABLE)
print("openrouter_rankings rows:", t.count(), "| days:", t.select("date").distinct().count(),
      "| range:", t.agg(F.min("date"), F.max("date")).collect()[0])
display(t.filter("date = (SELECT max(date) FROM " + TABLE + ") AND NOT is_aggregate").orderBy("rank").limit(10))

## Scheduling
One **weekly** Job (this notebook, **Source = Git provider / branch `main`**, serverless). The run-as
identity needs **READ** on the `frontier_labs` secret scope. Overwrites the full history each run.